In [ ]:

import torch
from torch import nn

import matplotlib.pyplot as plt

print(torch.__version__)


In [ ]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


In [ ]:

weight = 0.7
bias = 0.3

X = torch.arange(0, 1, 0.02).unsqueeze(dim=1)
y = weight * X + bias

X[:10], y[:10]


In [ ]:

split = int(0.8 * len(X))

X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

len(X_train), len(y_train), len(X_test), len(y_test)



In [ ]:

def plot_predictions(data_train=X_train,
                     label_train=y_train,
                     data_test=X_test,
                     label_test=y_test,
                     predictions=None):
    plt.figure(figsize=(10, 7))
    plt.scatter(data_train, label_train, c="b", s=4, label="Training Data")
    plt.scatter(data_test, label_test, c="g", s=4, label="Test Data")

    if predictions is not None:
        plt.scatter(data_test, predictions, c="r", s=4, label="Predictions")

    plt.legend(prop={"size": 14})

plot_predictions()

In [ ]:

class LinearRegressionModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_layer = nn.Linear(in_features=1,
                                      out_features=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor: 
        return self.linear_layer(x)

torch.manual_seed(42)
model_1 = LinearRegressionModelV2()
model_1, model_1.state_dict()


In [ ]:


next(model_1.parameters()).device



In [ ]:

loss_fn = nn.L1Loss()
optimizer = torch.optim.SGD(model_1.parameters(), lr=0.01)


In [ ]:

torch.manual_seed(42)

epochs = 1000

X_train = X_train.to(device)
y_train = y_train.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

for epoch in range(epochs):

    model_1.train()

    y_pred = model_1(X_train)
    loss = loss_fn(y_pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model_1.eval()

    with torch.inference_mode():
        test_pred = model_1(X_test)

        test_loss = loss_fn(test_pred, y_test)

    if epoch % 100 == 0:
        print(f"Epoch: {epoch} | Train Loss: {loss} | Test Loss: {test_loss}")



In [ ]:
from pprint import pprint

print("The model learned the following values for the weights and bias:")
pprint(model_1.state_dict())
print("\nAnd the original values for the weight and bias are:")
print(f"weights: {weight}, bias: {bias}")



In [ ]:

model_1.eval()


with torch.inference_mode():
    y_pred = model_1(X_test)

    y_idk = 0.6968 * X_test + 0.3025

    plot_predictions(predictions=y_pred)


In [ ]:

from pathlib import Path

MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "01_pytorch_workflow_model_1.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_1.state_dict(), f=MODEL_SAVE_PATH)


In [ ]:


model_2 = LinearRegressionModelV2()

model_2.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

model_2.to(device)

#print(f"Loaded model: \n {model_2}")
#print(f"Model on device: {next(model_2.parameters()).device}")

model_2.eval()

with torch.inference_mode():
    new_preds = model_2(X_test)

new_preds == y_pred

